# W9C1 Lab: Byte Pair Encoding, by Hand and for Real

Run every cell from the top. **Everything already works.**

Today you will:

1. See how a real tokenizer splits words you have never seen.
2. Run BPE merges yourself and watch the vocabulary build itself.
3. Find out what tokenization costs you on unusual text.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, logging

logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
print("vocabulary size:", f"{tokenizer.vocab_size:,}")

## Part 1. What a real tokenizer does to your words

A model has a fixed vocabulary. Any word outside it gets chopped into
pieces that ARE in it. Common words survive whole; rare ones do not.

In [ ]:
# GIVEN. Six words, from ordinary to obscure.
WORDS = ["the", "running", "tokenization", "antidisestablishmentarianism",
         "hyperparameter", "Trinity"]

rows = []
for w in WORDS:
    pieces = tokenizer.tokenize(" " + w)     # leading space: GPT-2 marks word starts
    rows.append({"word": w, "pieces": len(pieces), "split as": " | ".join(pieces)})

table = pd.DataFrame(rows)
print(table.to_string(index=False))

plt.figure(figsize=(7, 3))
plt.bar(table["word"], table["pieces"], color="#7C2529")
plt.ylabel("tokens used"); plt.xticks(rotation=35, ha="right")
plt.title("One word is not one token"); plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 1 ==================
# Find a word this tokenizer handles badly. Try your own name, a
# technical term, a word in another language, or an emoji.
#
# Then find a long word it handles in ONE token.
#
# Expected: common English words cost 1 token however long they are. Rare names,
#           non-English words and emoji cost 3 to 10. This is why an API bill is
#           not proportional to how much you wrote: it is proportional to how
#           ORDINARY what you wrote was.
# ===============================================
MY_WORDS = ["Trinity", "davalos", "Schadenfreude", "kwaheri", "photosynthesis"]

for w in MY_WORDS:
    pieces = tokenizer.tokenize(" " + w)
    print(f"   {w:<20} {len(pieces):>2} tokens   {' | '.join(pieces)}")

## Part 2. Building the vocabulary yourself

BPE starts with single characters and repeatedly glues together the
commonest adjacent pair. That is the entire algorithm.

In [ ]:
# GIVEN. Five merges on a tiny corpus, printed as they happen.
CORPUS = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

# every word starts as its characters, with a marker for the end of the word
words = {" ".join(w) + " </w>": n for w, n in CORPUS.items()}

def commonest_pair(words):
    pairs = Counter()
    for word, freq in words.items():
        symbols = word.split()
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += freq
    return pairs.most_common(1)[0] if pairs else (None, 0)

print("starting point:")
for w in words:
    print("   ", w)
print()

merges = []
for step in range(5):
    (pair, count) = commonest_pair(words)
    if pair is None:
        break
    merges.append(pair)
    joined = "".join(pair)
    words = {w.replace(" ".join(pair), joined): n for w, n in words.items()}
    print(f"merge {step + 1}: {pair[0]!r} + {pair[1]!r} -> {joined!r}  (seen {count} times)")

print()
print("after five merges:")
for w in words:
    print("   ", w)

In [ ]:
# ================== YOUR TURN 2 ==================
# Change the number of merges and watch the vocabulary change.
#
# Try 2, then 10, then 30.
#
# Expected: with few merges everything stays as characters, so sequences are long.
#           With many merges whole words become single tokens and sequences get
#           short. Vocabulary size and sequence length trade directly against each
#           other, and choosing where to sit on that trade IS the design decision.
# ===============================================
N_MERGES = 5          # <-- try 2, then 10, then 30

w = {" ".join(k) + " </w>": v for k, v in CORPUS.items()}
vocabulary = set()
for step in range(N_MERGES):
    pair, count = commonest_pair(w)
    if pair is None:
        print(f"ran out of pairs to merge after {step} merges")
        break
    w = {word.replace(" ".join(pair), "".join(pair)): n for word, n in w.items()}
    vocabulary.add("".join(pair))

lengths = [len(word.split()) for word in w]
print(f"merges: {N_MERGES}")
print(f"   new symbols learned: {len(vocabulary)}")
print(f"   average tokens per word: {sum(lengths) / len(lengths):.2f}")
for word in w:
    print("   ", word)

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Common English words are one token no matter how long. Names, non-English
#   words and emoji cost several. Two consequences you will meet again: prompts
#   in other languages cost more money for the same meaning, and a model's
#   context window holds less text when that text is unusual.
#
# YOUR TURN 2
#   Few merges means character-level: a tiny vocabulary and very long sequences.
#   Many merges means word-level: a huge vocabulary and short sequences. BPE is
#   the dial between those two, and every model picks a point on it (GPT-2 uses
#   about 50,000 merges).